<span style="color: rgb(99, 100, 102); font-family: Roboto, sans-serif; background-color: rgb(255, 255, 255);"> Township is built upon farming and production puzzle cores, and casual order board games. The player is harvesting crops such as wheat, corn, carrot, potato, sugarcane, cocoa, tomato, rubber, silk, strawberries, rice and pepper. Assets are used to produce goods in factories to earn coins and experience points.

[Source](https://en.wikipedia.org/wiki/Township_(video_game))<span style="background-color: rgb(255, 255, 255);"><br></span>

See the first 1000 rows of items, the production time and the constraint they depend upon.

In [1]:
SELECT TOP (1000) 
      [portfolio].[township].[items].[Id] as itemId
      , [portfolio].[township].[items].[name] as itemName
      , [portfolio].[township].[items].[productiontime] as productionTime
      , [portfolio].[township].[constraints].name as constraintName
  FROM [portfolio].[township].[items]
  JOIN [portfolio].[township].[constraints] ON [portfolio].[township].[items].[constraintId] = [portfolio].[township].[constraints].[Id]
  ORDER BY productionTime
  , constraintName; 
GO

(48 rows affected)

Total execution time: 00:00:00.151

itemId,itemName,productionTime,constraintName
1,gold,0,none
2,wheat,2,field
23,cow feed,4,feed mill
14,bread,5,bakery
3,corn,5,field
24,chicken feed,8,feed mill
4,carrot,10,field
27,cream,11,dairy factory
15,cookies,15,bakery
25,sheep feed,16,feed mill


List the items, their constraints, the production time and their dependancies.

In [2]:
SELECT TOP (1000) 
    [portfolio].[township].[items].[Id],
    [portfolio].[township].[items].[name] as [itemName],
    [portfolio].[township].[constraints].[name] AS [constraintName],
    [portfolio].[township].[items].[productiontime] as [productionTime],
    parentDetails.name AS [parentName],
    [portfolio].[township].[dependancies].[items] AS [numberOfItems]
  FROM [portfolio].[township].[items]
  JOIN [portfolio].[township].[constraints] ON [portfolio].[township].[items].[constraintId] = [portfolio].[township].[constraints].[Id]
  LEFT JOIN [portfolio].[township].[dependancies] ON [portfolio].[township].[items].[Id] = [portfolio].[township].[dependancies].[itemId]
  JOIN (SELECT * FROM [portfolio].[township].[items]) as parentDetails ON [portfolio].[township].[dependancies].[parentId] = parentDetails.Id
  WHERE [portfolio].[township].[dependancies].[parentId] IS NULL

UNION ALL

SELECT TOP (1000) 
    [portfolio].[township].[items].[Id],
    [portfolio].[township].[items].[name] as [itemName],
    [portfolio].[township].[constraints].[name] AS [constraintName],
    [portfolio].[township].[items].[productiontime] as [productionTime],
    parentDetails.name AS [parentName],
    [portfolio].[township].[dependancies].[items] AS [numberOfItems]
  FROM [portfolio].[township].[items]
  JOIN [portfolio].[township].[constraints] ON [portfolio].[township].[items].[constraintId] = [portfolio].[township].[constraints].[Id]
  LEFT JOIN [portfolio].[township].[dependancies] ON [portfolio].[township].[items].[Id] = [portfolio].[township].[dependancies].[itemId]
  JOIN (SELECT * FROM [portfolio].[township].[items]) as parentDetails ON [portfolio].[township].[dependancies].[parentId] = parentDetails.Id
  WHERE [portfolio].[township].[dependancies].[parentId] IS NOT NULL
  ORDER BY productionTime, itemName;
GO


(67 rows affected)

Total execution time: 00:00:00.094

Id,itemName,constraintName,productionTime,parentName,numberOfItems
2,wheat,field,2,gold,0
23,cow feed,feed mill,4,wheat,2
23,cow feed,feed mill,4,corn,1
14,bread,bakery,5,wheat,2
3,corn,field,5,gold,1
24,chicken feed,feed mill,8,wheat,2
24,chicken feed,feed mill,8,carrot,1
4,carrot,field,10,gold,2
27,cream,dairy factory,11,milk,1
15,cookies,bakery,15,wheat,2


list all the constraints and the dependent items ordered by production time.

In [3]:
SELECT TOP (1000) 
    [portfolio].[township].[constraints].[Id] as constriantId
    , [portfolio].[township].[constraints].[name] as constraintName
    ,[portfolio].[township].[items].[Id]
    ,[portfolio].[township].[items].[name] as itemName
    ,[portfolio].[township].[items].[productiontime] as productionTime
  FROM [portfolio].[township].[constraints]
  JOIN [portfolio].[township].[items] on [portfolio].[township].[constraints].[Id] = [portfolio].[township].[items].[constraintId]
  ORDER BY constriantId, productionTime, itemName;

(48 rows affected)

Total execution time: 00:00:00.045

constriantId,constraintName,Id,itemName,productionTime
1,apiary,22,honeycombs,360
2,bakery,14,bread,5
2,bakery,15,cookies,15
2,bakery,16,bagel,29
2,bakery,18,potato bread,57
2,bakery,17,pizza,114
3,chicken coop,19,eggs,60
4,cowshed,20,milk,20
5,dairy factory,27,cream,11
5,dairy factory,28,cheese,27


List all posssible product combinations per constraint under the productuion time limit.

In [4]:
DECLARE @constraintId INT;
DECLARE @productionTimeLimit INT = 60;

-- Declare a cursor to iterate through each constraintId
DECLARE constraint_cursor CURSOR FOR
SELECT DISTINCT constraintId
FROM portfolio.township.items
WHERE constraintId NOT IN (1);

-- Open the cursor
OPEN constraint_cursor;

-- Fetch the first constraintId
FETCH NEXT FROM constraint_cursor INTO @constraintId;

-- Loop through each constraintId
WHILE @@FETCH_STATUS = 0
BEGIN
    -- Print the constraintId (for demonstration purposes)
    PRINT 'Processing constraintId: ' + CAST(@constraintId AS VARCHAR);
    -- Run the RecursiveCTE query for the current constraintId
    WITH RecursiveCTE AS (
    SELECT
        [portfolio].[township].[items].[constraintId]
        ,CAST([portfolio].[township].[items].[Id] AS VARCHAR(MAX)) AS Combination
        ,CAST([portfolio].[township].[items].[name] AS VARCHAR(MAX)) AS [description]
        ,[portfolio].[township].[items].[productiontime] AS totalProductionTime
    FROM [portfolio].[township].[items]
    WHERE constraintId = @constraintId 
    AND productiontime <= @productionTimeLimit

    UNION ALL

    SELECT
        i.constraintId
        ,rc.Combination + ',' + CAST(i.[Id] AS VARCHAR(MAX))
        ,rc.[description] + ',' + CAST(i.[name] AS VARCHAR(MAX))
        ,rc.totalProductionTime + i.productiontime
    FROM RecursiveCTE rc
    JOIN [portfolio].[township].[items] i ON i.Id > CAST(SUBSTRING(rc.Combination, LEN(rc.Combination) - CHARINDEX(',', REVERSE(rc.Combination)) + 2, LEN(rc.Combination)) AS INT)
    WHERE
        i.constraintId = @constraintId 
    AND
        rc.totalProductionTime + i.[productiontime] <= @productionTimeLimit
    AND NOT EXISTS (
            SELECT 1
            FROM [portfolio].[township].[items] ni
            WHERE ni.Id = i.Id
            AND ',' + rc.Combination + ',' LIKE '%,' + CAST(ni.Id AS VARCHAR(MAX)) + ',%'
        )
    )
    SELECT
        constraintId
        ,c.name
        ,Combination
        ,[description]
        ,totalProductionTime
    FROM
        RecursiveCTE
    JOIN 
        [portfolio].[township].[constraints] c ON c.Id = RecursiveCTE.constraintId
    ORDER BY
        constraintId, totalProductionTime DESC;
    -- Fetch the next constraintId
    FETCH NEXT FROM constraint_cursor INTO @constraintId;
END
-- Close and deallocate the cursor
CLOSE constraint_cursor;
DEALLOCATE constraint_cursor;

Processing constraintId: 2

(13 rows affected)

Processing constraintId: 3

(1 row affected)

Processing constraintId: 4

...

constraintId,name,Combination,description,totalProductionTime
11,none,1,gold,0


constraintId,name,Combination,description,totalProductionTime
12,pastry factory,45,cupcake,57
12,pastry factory,44,brownie,38
12,pastry factory,43,muffin,30


constraintId,name,Combination,description,totalProductionTime
13,rubber factory,39,rubber,60


constraintId,name,Combination,description,totalProductionTime


constraintId,name,Combination,description,totalProductionTime
16,sugar factory,"32,31","syrup,sugar",60
16,sugar factory,"31,32","sugar,syrup",60
16,sugar factory,32,syrup,40
16,sugar factory,31,sugar,20


list all the items, they dependancies, production time and constraints

In [5]:
SELECT TOP (1000) [itemId]
      ,[parentId]
      ,p.name AS parentName
      ,[items]
      ,i.name AS itemName
      ,i.productiontime as productionTime
      ,c.name AS constraintName
  FROM [portfolio].[township].[dependancies] d
  JOIN [portfolio].[township].[items] i on i.Id = d.itemId
  JOIN [portfolio].[township].[items] p on p.Id = d.parentId
  JOIN [portfolio].[township].[constraints] c on c.Id = i.constraintId
  order by parentName, productionTime

(67 rows affected)

Total execution time: 00:00:00.044

itemId,parentId,parentName,items,itemName,productionTime,constraintName
46,16,bagel,1,donut,85,pastry factory
47,16,bagel,1,cheescake,171,pastry factory
22,26,bee feed,1,honeycombs,360,apiary
44,29,butter,1,brownie,38,pastry factory
44,11,cacao,2,brownie,38,pastry factory
46,11,cacao,1,donut,85,pastry factory
46,33,caramel,1,donut,85,pastry factory
24,4,carrot,1,chicken feed,8,feed mill
25,4,carrot,2,sheep feed,16,feed mill
17,28,cheese,1,pizza,114,bakery


List all the parrents and their dependancies

In [6]:
WITH RecursiveCTE AS (
    SELECT 
        d.itemId
        ,d.parentId
        ,p.name AS parentName
        ,i.name AS itemName
        ,i.productiontime AS productionTime
        ,c.name AS constraintName
        ,d.items
        , 1 as [level]
    FROM 
        portfolio.township.dependancies d
    JOIN 
        portfolio.township.items i ON i.Id = d.itemId
    JOIN 
        portfolio.township.items p ON p.Id = d.parentId
    JOIN 
        portfolio.township.constraints c ON c.Id = i.constraintId
    WHERE 
        i.name = 'pizza'

    UNION ALL

    SELECT 
        d.itemId
        ,d.parentId
        ,p.name AS parentName
        ,i.name AS itemName
        ,i.productiontime AS productionTime
        ,c.name AS constraintName
        ,d.items
        , rc.[level]+1 as [level]
    FROM 
        portfolio.township.dependancies d
    JOIN 
        portfolio.township.items i ON i.Id = d.itemId
    JOIN 
        portfolio.township.items p ON p.Id = d.parentId
    JOIN 
        portfolio.township.constraints c ON c.Id = i.constraintId
    JOIN 
        RecursiveCTE rc ON rc.parentId = d.itemId
)
SELECT 
    * 
FROM 
    RecursiveCTE
ORDER BY 
    [level] desc, parentName, productionTime;

(11 rows affected)

Total execution time: 00:00:00.095

itemId,parentId,parentName,itemName,productionTime,constraintName,items,level
2,1,gold,wheat,2,field,0,5
3,1,gold,corn,5,field,1,5
23,3,corn,cow feed,4,feed mill,1,4
23,2,wheat,cow feed,4,feed mill,2,4
20,23,cow feed,milk,20,cowshed,1,3
2,1,gold,wheat,2,field,0,2
8,1,gold,tomato,120,field,6,2
28,20,milk,cheese,27,dairy factory,2,2
17,28,cheese,pizza,114,bakery,1,1
17,8,tomato,pizza,114,bakery,2,1


In [8]:
DECLARE @itemName VARCHAR(50);

-- Declare a cursor to iterate through each constraintId
DECLARE dependancy_cursor CURSOR FOR
SELECT DISTINCT [name]
FROM portfolio.township.items
WHERE [name] NOT IN ('pizza');

-- Open the cursor
OPEN dependancy_cursor;

-- Fetch the first constraintId
FETCH NEXT FROM dependancy_cursor INTO @itemName;

-- Loop through each @itemName
WHILE @@FETCH_STATUS = 0
BEGIN
    -- Print the constraintId (for demonstration purposes)
    PRINT 'Processing item: ' + CAST(@itemName AS VARCHAR);
    -- Run the RecursiveCTE query for the current @itemName
    WITH RecursiveCTE AS (
        SELECT 
            d.itemId
            ,d.parentId
            ,p.name AS parentName
            ,i.name AS itemName
            ,i.productiontime AS productionTime
            ,c.name AS constraintName
            ,d.items AS requiredItems
            , 1 as [level]
        FROM 
            portfolio.township.dependancies d
        JOIN 
            portfolio.township.items i ON i.Id = d.itemId
        JOIN 
            portfolio.township.items p ON p.Id = d.parentId
        JOIN 
            portfolio.township.constraints c ON c.Id = i.constraintId
        WHERE 
            i.name = @itemName

        UNION ALL

        SELECT 
            d.itemId
            ,d.parentId
            ,p.name AS parentName
            ,i.name AS itemName
            ,i.productiontime AS productionTime
            ,c.name AS constraintName
            ,d.items AS requiredItems
            , rc.[level]+1 as [level]
        FROM 
            portfolio.township.dependancies d
        JOIN 
            portfolio.township.items i ON i.Id = d.itemId
        JOIN 
            portfolio.township.items p ON p.Id = d.parentId
        JOIN 
            portfolio.township.constraints c ON c.Id = i.constraintId
        JOIN 
            RecursiveCTE rc ON rc.parentId = d.itemId
    )
    SELECT 
        itemId
        , itemName
        , parentId
        , parentName as ingredientsName
        , constraintName
        , requiredItems
        , productionTime
        , requiredItems * productionTime as totalProductionTime
        , requiredItems * (SELECT productiontime FROM portfolio.township.items itm where parentId = itm.Id) as totalProductionTime1
    FROM 
        RecursiveCTE
    ORDER BY 
        [level] desc, parentName, productionTime;
    -- Fetch the next constraintId
    FETCH NEXT FROM dependancy_cursor INTO @itemName;
END
-- Close and deallocate the cursor
CLOSE dependancy_cursor;
DEALLOCATE dependancy_cursor;

Processing item: bagel

(11 rows affected)

Processing item: bee feed

(4 rows affected)

Processing item: bread

...

itemId,itemName,parentId,ingredientsName,constraintName,requiredItems,productionTime,totalProductionTime,totalProductionTime1
8,tomato,1,gold,field,6,120,720,0


itemId,itemName,parentId,ingredientsName,constraintName,requiredItems,productionTime,totalProductionTime,totalProductionTime1


itemId,itemName,parentId,ingredientsName,constraintName,requiredItems,productionTime,totalProductionTime,totalProductionTime1
2,wheat,1,gold,field,0,2,0,0


itemId,itemName,parentId,ingredientsName,constraintName,requiredItems,productionTime,totalProductionTime,totalProductionTime1
3,corn,1,gold,field,1,5,5,0
4,carrot,1,gold,field,2,10,20,0
25,sheep feed,4,carrot,feed mill,2,16,32,20
25,sheep feed,3,corn,feed mill,2,16,32,10
21,wool,25,sheep feed,sheep farm,1,240,240,16


itemId,itemName,parentId,ingredientsName,constraintName,requiredItems,productionTime,totalProductionTime,totalProductionTime1
2,wheat,1,gold,field,0,2,0,0
3,corn,1,gold,field,1,5,5,0
23,cow feed,3,corn,feed mill,1,4,4,5
23,cow feed,2,wheat,feed mill,2,4,8,4
20,milk,23,cow feed,cowshed,1,20,20,4
30,yogurt,20,milk,dairy factory,4,81,324,80
